# System 1 build release

Streams every worker and caption package of a release, one video at a time, into the
runtime release for System 2 (SQLite with FTS5, image and text indexes, WebP media) under
`release_build/release`. With `PUBLISH`, each video's media is uploaded to the bucket during
the build, then `db/`, `indexes/`, `manifest.json`, and `READY.json` last.

Edit only the settings cell, then Run All.

In [ ]:
# Settings
REPO_URL = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
REPO_REF = "dev"
RELEASE_ID = "canonical_release_v001"
SOURCE_REPO = "1thesudden/AIOU26_release_test"  # repo that holds the worker and caption packages
BUCKET_ID = "1thesudden/aiou-app-storage"
BUCKET_PREFIX = "releases_test"  # production: "releases"
PUBLISH = True

In [ ]:
# Setup: secrets, repository, and the worker extra. Runs on a Kaggle CPU kernel (about 30 GB RAM) or Colab; no GPU needed.
import os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

token = secret("HF_TOKEN")
if not token:
    raise RuntimeError("Add an HF_TOKEN secret with write access to the output repo")
os.environ["HF_TOKEN"] = token  # never printed

WORK = Path("/kaggle/temp" if Path("/kaggle").exists() else "/content")
REPO_DIR = WORK / "mare"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--quiet", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR / 'system1'}[release]"], check=True)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip())


In [ ]:
def system1(*args):
    command = [sys.executable, "-m", "system1.cli", *map(str, args)]
    with subprocess.Popen(command, cwd=REPO_DIR / "system1", stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
    if process.returncode:
        raise RuntimeError(f"system1 exited with {process.returncode}")

PERSIST = Path("/kaggle/working" if Path("/kaggle").exists() else "/content") / "build_release_report"
system1("build-release", "--release-id", RELEASE_ID, "--source-repo", SOURCE_REPO,
        "--out-dir", WORK / "release_build", "--bucket-id", BUCKET_ID, "--bucket-prefix", BUCKET_PREFIX,
        "--publish" if PUBLISH else "--no-publish", "--persist-dir", PERSIST)